<a href="https://colab.research.google.com/github/kursatkara/MAE_5020_Spring_2026/blob/master/05_07_exact_DMD_unknown_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAE 5020 Guided Example: Exact DMD on Unknown Data from a File

In the previous class, we studied exact Dynamic Mode Decomposition (DMD) on a toy dataset with known latent dynamics.  
In this notebook, you will apply the **same exact DMD workflow** to a dataset whose dynamics are **not revealed in advance**.

## Learning goals

By the end of this notebook, you should be able to:

1. load measurement data from an `.npz` file,
2. inspect the data before doing any modeling,
3. build the snapshot matrices $X_1$ and $X_2$,
4. perform exact DMD step by step,
5. choose and justify a truncation rank,
6. interpret eigenvalues in terms of growth/decay and oscillation frequency,
7. analyze DMD modes and reconstruction quality,
8. discuss what features appear trustworthy and what features may be caused by noise.

## What is in the data file?

The file contains:

- `X`: a spatially distributed dataset with shape `(n_space, n_snapshots)`,
- `t`: the time vector,
- `x`: the spatial grid,
- `dt`: the time step.

You should treat the dataset as **unknown**. Your task is to use DMD to extract as much information as possible from it.

## 1. Imports and plotting settings

We will use only standard scientific Python tools available in Google Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.dpi"] = 120

## 2. Load the unknown dataset from a file

This notebook is designed to work with the file:

`mae5020_dmd_unknown_case.npz`

### In Google Colab
Upload the `.npz` file when prompted.

### If the file is already in the working directory
The notebook will load it directly.

In [ ]:
import os

filename = "mae5020_dmd_unknown_case.npz"

if not os.path.exists(filename):
    try:
        from google.colab import files
        print(f"Please upload {filename}")
        uploaded = files.upload()
    except Exception as e:
        raise FileNotFoundError(
            f"{filename} was not found. Upload it to Colab or place it in the working directory."
        ) from e

data = np.load(filename)

print("Available keys in the file:")
print(list(data.keys()))

In [ ]:
X = data["X"]
t = data["t"]
x = data["x"]
dt = float(data["dt"])

n_space, n_snap = X.shape

print(f"X shape       : {X.shape}")
print(f"t shape       : {t.shape}")
print(f"x shape       : {x.shape}")
print(f"dt            : {dt:.6f}")
print(f"time span     : {t[0]:.3f} to {t[-1]:.3f}")
print(f"spatial extent: {x[0]:.3f} to {x[-1]:.3f}")

## 3. First look at the dataset

Before running DMD, always inspect the data.

### Questions to think about
- Does the field appear oscillatory, decaying, growing, or drifting?
- Are there localized spatial features?
- Does the data look noisy?
- Do you expect a low-rank structure?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(
    X,
    aspect="auto",
    origin="lower",
    extent=[t[0], t[-1], x[0], x[-1]]
)
ax.set_xlabel("Time")
ax.set_ylabel("Spatial coordinate")
ax.set_title("Unknown data field X(x,t)")
plt.colorbar(im, ax=ax, label="Signal value")
plt.show()

In [ ]:
snapshot_ids = [0, 1, 2, 3, 4]

plt.figure(figsize=(8,4))
for k in snapshot_ids:
    plt.plot(x, X[:, k], label=f"k={k}, t={t[k]:.2f}")
plt.xlabel("x")
plt.ylabel("Field value")
plt.title("Selected spatial snapshots")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
snapshot_ids = [0, n_snap//4, n_snap//2, 3*n_snap//4, n_snap-1]

plt.figure(figsize=(8,4))
for k in snapshot_ids:
    plt.plot(x, X[:, k], label=f"k={k}, t={t[k]:.2f}")
plt.xlabel("x")
plt.ylabel("Field value")
plt.title("Selected spatial snapshots")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Build the snapshot matrices

As in exact DMD, define

$$
X_1 = \begin{bmatrix}
x_1 & x_2 & \cdots & x_{m-1}
\end{bmatrix},
\qquad
X_2 = \begin{bmatrix}
x_2 & x_3 & \cdots & x_m
\end{bmatrix}.
$$

Here each column is one snapshot of the measured field.

In [ ]:
X1 = X[:, :-1]
X2 = X[:, 1:]

print("X1 shape:", X1.shape)
print("X2 shape:", X2.shape)

## 5. Compute the SVD of $X_1$

The singular values help us estimate an effective rank and judge whether the data may be noisy.

In [ ]:
U, s, Vh = np.linalg.svd(X1, full_matrices=False)
V = Vh.T

print("First 10 singular values:")
print(s[:10])

In [ ]:
energy = s**2
cum_energy = np.cumsum(energy) / np.sum(energy)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].semilogy(np.arange(1, len(s)+1), s, "o-")
#ax[0].loglog(np.arange(1, len(s)+1), s, "o-")
ax[0].set_xlabel("Index")
ax[0].set_ylabel("Singular value")
ax[0].set_title("Singular values of $X_1$")
ax[0].grid(True, alpha=0.3)

ax[1].plot(np.arange(1, len(s)+1), cum_energy, "o-")
#ax[1].loglog(np.arange(1, len(s)+1), cum_energy, "o-")
ax[1].axhline(0.80, color="r", linestyle="--", label="80% energy")
ax[1].axhline(0.90, color="g", linestyle="--", label="90% energy")
ax[1].set_xlabel("Rank")
ax[1].set_ylabel("Cumulative energy")
ax[1].set_title("Cumulative energy")
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()

## 6. Choose a truncation rank

Use the singular values and the cumulative energy plot to select a rank.

For this guided example, start with a rank that seems physically reasonable, then test nearby values later.

### Suggested task
- Write down your first guess for the rank.
- Explain why you chose it.

In [ ]:
# Try changing this value after inspecting the singular values.
r = 3

Ur = U[:, :r]
Sr = np.diag(s[:r])
Vr = V[:, :r]

print("Chosen rank r =", r)
print("Ur shape:", Ur.shape)
print("Sr shape:", Sr.shape)
print("Vr shape:", Vr.shape)

## 7. Form the reduced operator $\widetilde{A}$

The reduced exact DMD operator is

$$
\widetilde{A} = U_r^T X_2 V_r \Sigma_r^{-1}.
$$

This matrix is small, so its eigendecomposition is inexpensive.

In [ ]:
A_tilde = Ur.T @ X2 @ Vr @ np.linalg.inv(Sr)
print("A_tilde shape:", A_tilde.shape)
print(A_tilde)

## 8. Compute eigenvalues and exact DMD modes

Solve

$$
\widetilde{A} W = W \Lambda.
$$

Then compute the exact DMD modes

$$
\Phi = X_2 V_r \Sigma_r^{-1} W.
$$

In [ ]:
eigvals, W = np.linalg.eig(A_tilde)
Phi = X2 @ Vr @ np.linalg.inv(Sr) @ W

print("Discrete-time DMD eigenvalues:")
print(eigvals)
print("Phi shape:", Phi.shape)

## 9. Convert the eigenvalues to continuous-time quantities

From the discrete-time eigenvalues $\lambda_j$, define

$$
\mu_j = \frac{\log(\lambda_j)}{\Delta t}.
$$

Then

- $\Re(\mu_j)$ gives the growth or decay rate,
- $\Im(\mu_j)$ gives the angular frequency,
- $f_j = \Im(\mu_j)/(2\pi)$ gives the frequency in Hz.

In [ ]:
mu = np.log(eigvals) / dt
growth_rates = np.real(mu)
omegas = np.imag(mu)
freq_hz = omegas / (2*np.pi)

# Simple summary table
for j in range(r):
    print(
        f"Mode {j+1}: "
        f"lambda = {eigvals[j]: .6f}, "
        f"growth/decay = {growth_rates[j]: .6f}, "
        f"omega = {omegas[j]: .6f}, "
        f"f = {freq_hz[j]: .6f} Hz"
    )

In [ ]:
# Sort by modal amplitude later, but first show eigenvalues in the complex plane
theta = np.linspace(0, 2*np.pi, 400)

plt.figure(figsize=(5,5))
plt.plot(np.cos(theta), np.sin(theta), "k--", label="Unit circle")
plt.scatter(np.real(eigvals), np.imag(eigvals), s=80)
for j, lam in enumerate(eigvals):
    plt.text(np.real(lam)+0.01, np.imag(lam)+0.01, f"{j+1}")
plt.xlabel("Real")
plt.ylabel("Imaginary")
plt.title("DMD eigenvalues in the complex plane")
plt.axis("equal")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 10. Compute modal amplitudes

We approximate the initial snapshot using

$$
x_1 \approx \Phi b,
$$

so that

$$
b = \Phi^{+} x_1.
$$

These amplitudes help us identify which modes are dynamically important in the reconstruction.

In [ ]:
b = np.linalg.pinv(Phi) @ X[:, 0]
print("Modal amplitudes b:")
print(b)

In [ ]:
amp = np.abs(b)
order = np.argsort(-amp)

print("Modes sorted by modal amplitude:")
for idx in order:
    print(
        f"Mode {idx+1}: |b| = {amp[idx]:.6f}, "
        f"lambda = {eigvals[idx]:.6f}, "
        f"f = {freq_hz[idx]:.6f} Hz"
    )

## 11. Visualize the DMD modes

For real-valued datasets, oscillatory modes often appear as complex-conjugate pairs.  
Plot the real and imaginary parts to inspect the spatial structures.

### Questions to think about
- Which modes appear as a complex-conjugate pair?
- Is there a mostly real mode?
- Do the mode shapes appear smooth or noisy?
- Are some modes localized in space?

In [ ]:
for j in range(r):
    plt.figure(figsize=(8,3.5))
    plt.plot(x, np.real(Phi[:, j]), label="Real part")
    plt.plot(x, np.imag(Phi[:, j]), "--", label="Imag part")
    plt.xlabel("x")
    plt.ylabel("Mode value")
    plt.title(f"DMD mode {j+1}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## 12. Reconstruct the data from the DMD model

Using the discrete-time eigenvalues, the temporal dynamics are

$$
x_k \approx \Phi \Lambda^{k-1} b.
$$

We reconstruct all snapshots and compare the result to the data.

In [ ]:
time_dynamics = np.zeros((r, n_snap), dtype=complex)

for k in range(n_snap):
    time_dynamics[:, k] = b * (eigvals**k)

X_dmd = Phi @ time_dynamics
X_dmd_real = np.real(X_dmd)

rel_err = np.linalg.norm(X - X_dmd_real) / np.linalg.norm(X)
print(f"Relative reconstruction error = {rel_err:.6e}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

im0 = ax[0].imshow(
    X,
    aspect="auto",
    origin="lower",
    extent=[t[0], t[-1], x[0], x[-1]]
)
ax[0].set_title("Original data")
ax[0].set_xlabel("Time")
ax[0].set_ylabel("x")
plt.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(
    X_dmd_real,
    aspect="auto",
    origin="lower",
    extent=[t[0], t[-1], x[0], x[-1]]
)
ax[1].set_title("DMD reconstruction")
ax[1].set_xlabel("Time")
plt.colorbar(im1, ax=ax[1])

im2 = ax[2].imshow(
    X - X_dmd_real,
    aspect="auto",
    origin="lower",
    extent=[t[0], t[-1], x[0], x[-1]]
)
ax[2].set_title("Residual: data - DMD")
ax[2].set_xlabel("Time")
plt.colorbar(im2, ax=ax[2])

plt.tight_layout()
plt.show()

In [ ]:
snapshot_ids = [0, n_snap//4, n_snap//2, 3*n_snap//4, n_snap-1]

for k in snapshot_ids:
    plt.figure(figsize=(8,3.5))
    plt.plot(x, X[:, k], label="Data", linewidth=2)
    plt.plot(x, X_dmd_real[:, k], "--", label="DMD", linewidth=2)
    plt.xlabel("x")
    plt.ylabel("Field value")
    plt.title(f"Snapshot comparison at t = {t[k]:.2f}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## 13. Rank sensitivity study

A good DMD analysis should not depend entirely on one arbitrary choice of rank.

### Task
Try several ranks and compare:
- reconstruction error,
- eigenvalues,
- inferred frequencies,
- mode smoothness.

If the data are noisy, very large ranks may begin to fit noise instead of dynamics.

In [ ]:
candidate_ranks = range(2, 9)
errors = []
summaries = []

for rr in candidate_ranks:
    Ur = U[:, :rr]
    Sr = np.diag(s[:rr])
    Vr = V[:, :rr]

    A_tilde = Ur.T @ X2 @ Vr @ np.linalg.inv(Sr)
    eigvals_rr, W_rr = np.linalg.eig(A_tilde)
    Phi_rr = X2 @ Vr @ np.linalg.inv(Sr) @ W_rr
    b_rr = np.linalg.pinv(Phi_rr) @ X[:, 0]

    td_rr = np.zeros((rr, n_snap), dtype=complex)
    for k in range(n_snap):
        td_rr[:, k] = b_rr * (eigvals_rr**k)

    X_rr = np.real(Phi_rr @ td_rr)
    err_rr = np.linalg.norm(X - X_rr) / np.linalg.norm(X)

    errors.append(err_rr)
    mu_rr = np.log(eigvals_rr) / dt
    summaries.append((rr, eigvals_rr, np.real(mu_rr), np.imag(mu_rr)/(2*np.pi)))

plt.figure(figsize=(6,4))
plt.plot(list(candidate_ranks), errors, "o-")
plt.xlabel("Rank r")
plt.ylabel("Relative reconstruction error")
plt.title("Rank sensitivity of DMD reconstruction")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
for rr, eig_rr, growth_rr, freq_rr in summaries:
    print(f"\nRank r = {rr}")
    for j in range(rr):
        print(
            f"  Mode {j+1}: lambda = {eig_rr[j]: .5f}, "
            f"growth or decay = {growth_rr[j]: .5f}, "
            f"f = {freq_rr[j]: .5f} Hz"
        )

## 14. What can you infer from the unknown data?

Use your DMD results to answer the following.

### Suggested interpretation questions
1. What is the likely effective rank of the dominant dynamics?
2. Do you see an oscillatory mode pair? If so, what is the dominant frequency?
3. Do the dominant dynamics grow, decay, or remain nearly neutral?
4. Is there evidence of a non-oscillatory mode?
5. Which mode shapes appear physically meaningful?
6. Which features may be caused by measurement noise or overfitting?
7. Which truncation rank seems most defensible, and why?

### Write your interpretation here

Replace this text with your own conclusions after running the notebook.

## 15. Optional extension: smooth surrogate fits for recovered modes

DMD returns sampled mode vectors on the measurement grid.  
If a mode looks physically meaningful but noisy, one can try fitting a smooth surrogate.

Below is one optional example using a low-order Fourier basis.  
This does **not** change the DMD result. It only helps interpret a recovered mode shape.

In [ ]:
# Pick the mode with the largest amplitude
j_star = order[0]
phi_star = np.real(Phi[:, j_star])

# Build a simple truncated Fourier basis
n_harm = 4
basis = [np.ones_like(x)]
labels = ["1"]
for k in range(1, n_harm + 1):
    basis.append(np.sin(2*np.pi*k*x))
    labels.append(f"sin(2π{k}x)")
    basis.append(np.cos(2*np.pi*k*x))
    labels.append(f"cos(2π{k}x)")

Psi = np.column_stack(basis)
coef, *_ = np.linalg.lstsq(Psi, phi_star, rcond=None)
phi_fourier = Psi @ coef

plt.figure(figsize=(8,4))
plt.plot(x, phi_star, label="Recovered DMD mode")
plt.plot(x, phi_fourier, "--", label="Smooth Fourier fit")
plt.xlabel("x")
plt.ylabel("Mode value")
plt.title("Optional smooth surrogate fit")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("Largest Fourier coefficients:")
for c, name in sorted(zip(coef, labels), key=lambda z: -abs(z[0]))[:6]:
    print(f"{name:10s}: {c:+.5f}")

## 16. Final takeaway

When the governing equations are unknown, exact DMD can still reveal:

- a likely low-dimensional dynamical structure,
- dominant oscillation frequencies,
- growth or decay rates,
- spatial mode shapes,
- reconstruction quality,
- and sensitivity of the conclusions to rank choice and noise.

That is exactly why DMD is so useful in data-driven analysis of dynamical systems.